#### Select a chat model

👉 Read the [OpenAI chat model integration docs](https://docs.langchain.com/oss/python/integrations/chat/openai/)

```shell
!pip install "langchain[openai]"
```

In [1]:
!pip install "langchain[openai]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.0/503.0 kB 23.1 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.17
    Uninstalling langchain-core-1.2.17:
      Successfully uninstalled langchain-core-1.2.17


In [3]:
import os
from google.colab import userdata

# We use OpenRouter for the agent — add OPENROUTER_API_KEY to Colab Secrets (key icon in left sidebar)
# Get your key at https://openrouter.ai/keys
os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")

In [4]:
from langchain_openai import ChatOpenAI

# https://openrouter.ai/nvidia/nemotron-3-nano-30b-a3b:free
model_nemotron3_nano = ChatOpenAI(
    model="nvidia/nemotron-3-nano-30b-a3b:free",
    temperature=0,
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY"),
)

## Authenticate Google Access

In [8]:
import os
# THIS IS THE FIX: Tell OAuth to allow http://localhost
os.environ['OAUTHLIB_INSECURE_TRANSPORT'] = '1'

from google_auth_oauthlib.flow import Flow
from googleapiclient.discovery import build

SCOPES = [
    'https://www.googleapis.com/auth/calendar',
    'https://www.googleapis.com/auth/gmail.modify'
]

# 1. Load the credentials file
try:
    flow = Flow.from_client_secrets_file('credentials.json', scopes=SCOPES, redirect_uri='http://localhost:8080/')
except FileNotFoundError:
    print("❌ ERROR: 'credentials.json' not found. Please upload it!")
    raise

# 2. Generate the login link
auth_url, _ = flow.authorization_url(prompt='consent')
print("\n" + "="*60)
print("1. Click this link and log in:")
print(auth_url)
print("="*60 + "\n")

# 3. Get the localhost URL from you
redirect_response = input("Paste the ENTIRE http://localhost... URL here and press Enter: ")

# 4. Exchange for tokens
flow.fetch_token(authorization_response=redirect_response.strip())
creds = flow.credentials

calendar_service = build('calendar', 'v3', credentials=creds)
gmail_service = build('gmail', 'v1', credentials=creds)

print("\n✅ SUCCESS! Google APIs are connected and ready.")


1. Click this link and log in:
https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=1084666946403-q0ui7h87prg3pk7a2ho77rfgqode8lv1.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8080%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcalendar+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fgmail.modify&state=UQwEBSXXJBYK5FYxwd9LaFIhBHe6PT&code_challenge=d4WDijLceuZtRqIqclrP8nB36P6sAv5WtQY-XVkcN9I&code_challenge_method=S256&prompt=consent&access_type=offline

Paste the ENTIRE http://localhost... URL here and press Enter: http://localhost:8080/?state=UQwEBSXXJBYK5FYxwd9LaFIhBHe6PT&iss=https://accounts.google.com&code=4/0AfrIepD1dPhr4vU2Xdf9FJ3sCxPTYxrXVD-VQeNHHIxLbI2Zz3cwnwJegP5p_HOfL7ePpg&scope=https://www.googleapis.com/auth/gmail.modify%20https://www.googleapis.com/auth/calendar

✅ SUCCESS! Google APIs are connected and ready.


## 1. Define tools

In [9]:
from langchain.tools import tool

In [10]:
import base64
from email.message import EmailMessage

@tool
def create_calendar_event(
    title: str,
    start_time: str,       # ISO format: "2024-01-15T14:00:00"
    end_time: str,         # ISO format: "2024-01-15T15:00:00"
    attendees: list[str],  # email addresses
    location: str = ""
) -> str:
    """Create a calendar event. Requires exact ISO datetime format."""
    try:
        event = {
            'summary': title,
            'location': location,
            'start': {'dateTime': start_time, 'timeZone': 'UTC'},
            'end': {'dateTime': end_time, 'timeZone': 'UTC'},
            'attendees': [{'email': email} for email in attendees],
        }
        # REAL API CALL to Google Calendar
        calendar_service.events().insert(calendarId='primary', body=event).execute()
        return f"Event created: {title} from {start_time} to {end_time} with {len(attendees)} attendees"
    except Exception as e:
        return f"Error creating event: {e}"

In [11]:
@tool
def send_email(
    to: list[str],  # email addresses
    subject: str,
    body: str,
    cc: list[str] = []
) -> str:
    """Send an email via email API. Requires properly formatted addresses."""
    try:
        message = EmailMessage()
        message.set_content(body)
        message['To'] = ', '.join(to)
        message['Subject'] = subject
        if cc:
            message['Cc'] = ', '.join(cc)

        encoded_message = base64.urlsafe_b64encode(message.as_bytes()).decode()
        create_message = {'raw': encoded_message}

        # REAL API CALL to Gmail
        gmail_service.users().messages().send(userId="me", body=create_message).execute()
        return f"Email sent to {', '.join(to)} - Subject: {subject}"
    except Exception as e:
        return f"Error sending email: {e}"

In [12]:
@tool
def get_available_time_slots(
    attendees: list[str],
    date: str,  # ISO format: "2024-01-15"
    duration_minutes: int
) -> list[str]:
    """Check calendar availability for given attendees on a specific date."""
    try:
        t_min = date + 'T00:00:00Z'
        t_max = date + 'T23:59:59Z'

        # REAL API CALL to check your actual schedule
        events_result = calendar_service.events().list(
            calendarId='primary', timeMin=t_min, timeMax=t_max, singleEvents=True
        ).execute()

        events = events_result.get('items', [])
        if not events:
            # If your calendar is completely empty that day, give the agent these options to pick from
            return ["09:00", "14:00", "16:00"]

        busy_times = [e['start'].get('dateTime') for e in events]
        return [f"I am busy at these times: {busy_times}. Please pick another time."]
    except Exception as e:
        return [f"Error checking calendar: {e}"]

## 2. Create specialized sub-agents

Next, we'll create specialized sub-agents that handle each domain.


### Create a calendar agent

- The calendar agent understands natural language scheduling requests and translates them into precise API calls.
- It handles date parsing, availability checking, and event creation.

In [17]:
import datetime
from langchain.agents import create_agent

# 1. Grab the real, actual current date and time
today = datetime.datetime.now().strftime("%A, %B %d, %Y")

# 2. Add it to the prompt so the AI knows exactly when "today" is
CALENDAR_AGENT_PROMPT = (
    f"You are a calendar scheduling assistant. Today is {today}. "
    "Parse natural language scheduling requests (e.g., 'next Tuesday at 2pm') "
    "into proper ISO datetime formats. "
    "Use get_available_time_slots to check availability when needed. "
    "Use create_calendar_event to schedule events. "
    "Always confirm what was scheduled in your final response."
)

calendar_agent = create_agent(
    model=model_nemotron3_nano,
    tools=[
        create_calendar_event,
        get_available_time_slots
    ],
    system_prompt=CALENDAR_AGENT_PROMPT,
)

Test the calendar agent to see how it handles natural language scheduling:

In [14]:
from langchain.messages import HumanMessage

In [18]:
query = "Schedule a team meeting with rrmalj@gmail.com next Tuesday at 2pm for 1 hour"
result1 = calendar_agent.invoke({"messages": [HumanMessage(query)]})
for message in result1.get("messages", []):
    message.pretty_print()

================================ Human Message =================================

Schedule a team meeting with rrmalj@gmail.com next Tuesday at 2pm for 1 hour
================================== Ai Message ==================================
Tool Calls:
  get_available_time_slots (call_9385fd1a08d0481194366f6b)
 Call ID: call_9385fd1a08d0481194366f6b
  Args:
    attendees: ['rrmalj@gmail.com']
    date: 2026-03-17
    duration_minutes: 60
================================= Tool Message =================================
Name: get_available_time_slots

["09:00", "14:00", "16:00"]
================================== Ai Message ==================================
Tool Calls:
  create_calendar_event (call_3720f10b56284f318ded0f2c)
 Call ID: call_3720f10b56284f318ded0f2c
  Args:
    end_time: 2026-03-17T15:00:00
    attendees: ['rrmalj@gmail.com']
    start_time: 2026-03-17T14:00:00
    title: Team Meeting
================================= Tool Message =================================
Name: crea

The agent parses "next Tuesday at 2pm" into ISO format ("2024-01-16T14:00:00"), calculates the end time, calls `create_calendar_event`, and returns a natural language confirmation.

### Create an email agent

- The email agent handles message composition and sending.
- It focuses on extracting recipient information, crafting appropriate subject lines and body text, and managing email communication.

In [19]:
EMAIL_AGENT_PROMPT = (
    "You are an email assistant. "
    "Compose professional emails based on natural language requests. "
    "Extract recipient information and craft appropriate subject lines and body text. "
    "Use send_email to send the message. "
    "Always confirm what was sent in your final response."
)

email_agent = create_agent(
    model=model_nemotron3_nano,
    tools=[send_email],
    system_prompt=EMAIL_AGENT_PROMPT,
)

Test the email agent with a natural language request:

In [21]:
query = "Send an email to rrmalj@gmail.com with a reminder about reviewing the new mockups"
result2 = email_agent.invoke({"messages": [HumanMessage(query)]})
for message in result2.get("messages", []):
    message.pretty_print()

================================ Human Message =================================

Send an email to rrmalj@gmail.com with a reminder about reviewing the new mockups
================================== Ai Message ==================================
Tool Calls:
  send_email (call_e2ea29a87ce741dda491cfe7)
 Call ID: call_e2ea29a87ce741dda491cfe7
  Args:
    body: Hi, please review the new mockups at your earliest convenience. Let me know if you have any feedback.
    subject: Reminder: Review New Mockups
    to: ['rrmalj@gmail.com']
================================= Tool Message =================================
Name: send_email

Email sent to rrmalj@gmail.com - Subject: Reminder: Review New Mockups
================================== Ai Message ==================================

The email with the subject "Reminder: Review New Mockups" has been successfully sent to rrmalj@gmail.com. Let me know if you'd like to follow up or need assistance with anything else!


## 3. Wrap sub-agents as tools

- Now wrap each sub-agent as a tool that the supervisor can invoke.
- This is the key architectural step that creates the layered system.
- The supervisor will see high-level tools like `"schedule_event"`, not low-level tools like `"create_calendar_event"`.

In [22]:
@tool
def schedule_event(request: str) -> str:
    """Schedule calendar events using natural language.

    Use this when the user wants to create, modify, or check calendar appointments.
    Handles date/time parsing, availability checking, and event creation.

    Input: Natural language scheduling request (e.g., 'meeting with design team
    next Tuesday at 2pm')
    """
    result = calendar_agent.invoke({"messages": [HumanMessage(request)]})
    return result["messages"][-1].text

In [23]:
@tool
def manage_email(request: str) -> str:
    """Send emails using natural language.

    Use this when the user wants to send notifications, reminders, or any email
    communication. Handles recipient extraction, subject generation, and email
    composition.

    Input: Natural language email request (e.g., 'send them a reminder about
    the meeting')
    """
    result = email_agent.invoke({"messages": [HumanMessage(request)]})
    return result["messages"][-1].text


- The **tool descriptions (docstring)** help the supervisor decide when to use each tool, so make them clear and specific.
- We return only the sub-agent's final response, as the supervisor doesn't need to see intermediate reasoning or tool calls.

## 4. Create the supervisor agent

- Now create the supervisor that orchestrates the sub-agents.
- The supervisor only sees high-level tools and makes routing decisions at the domain level, not the individual API level.

In [24]:
SUPERVISOR_PROMPT = (
    "You are a helpful personal assistant. "
    "You can schedule calendar events and send emails. "
    "Break down user requests into appropriate tool calls and coordinate the results. "
    "When a request involves multiple actions, use multiple tools in sequence."
)

supervisor_agent = create_agent(
    model=model_nemotron3_nano,
    tools=[
        schedule_event,
        manage_email
    ],
    system_prompt=SUPERVISOR_PROMPT,
)

## 5. Use the supervisor

Now test your complete system with complex requests that require coordination across multiple domains:


### Example 1: Simple single-domain request

In [25]:
query = "Schedule a team standup with rrmalj@gmail.com for tomorrow at 9am"
result3 = supervisor_agent.invoke({"messages": [HumanMessage(query)]})
for message in result3.get("messages", []):
    message.pretty_print()

================================ Human Message =================================

Schedule a team standup with rrmalj@gmail.com for tomorrow at 9am
================================== Ai Message ==================================
Tool Calls:
  schedule_event (call_ea5a928e92d8405aace2bb6c)
 Call ID: call_ea5a928e92d8405aace2bb6c
  Args:
    request: Schedule a team standup with rrmalj@gmail.com for tomorrow at 9am
================================= Tool Message =================================
Name: schedule_event

Your team standup has been scheduled:

- **Title:** Team Standup  
- **Date & Time:** Tomorrow, March 12, 2026, from 9:00 AM to 9:30 AM  
- **Attendee:** rrmalj@gmail.com  
- **Location:** Virtual  

Let me know if you’d like to adjust the time, duration, or add any other participants.
================================== Ai Message ==================================

Your team standup has been scheduled for **tomorrow at 9 am** with **rrmalj@gmail.com**. The event is set for a

The supervisor identifies this as a calendar task, calls `schedule_event`, and the calendar agent handles date parsing and event creation.

::: {.callout-tip}
For full transparency into the information flow, including prompts and responses for each chat model call, check out the [LangSmith trace](https://smith.langchain.com/public/91a9a95f-fba9-4e84-aff0-371861ad2f4a/r) for the above run.
:::

### Example 2: Complex multi-domain request

In [26]:
query = (
    "Schedule a meeting with rrmalj@gmail.com next Tuesday at 2pm for 1 hour, "
    "and send them an email reminder about reviewing the new mockups."
)
result4 = supervisor_agent.invoke({"messages": [HumanMessage(query)]})
for message in result4.get("messages", []):
    message.pretty_print()

================================ Human Message =================================

Schedule a meeting with rrmalj@gmail.com next Tuesday at 2pm for 1 hour, and send them an email reminder about reviewing the new mockups.
================================== Ai Message ==================================
Tool Calls:
  schedule_event (call_2b46c946ba1c46b8aab88314)
 Call ID: call_2b46c946ba1c46b8aab88314
  Args:
    request: meeting with rrmalj@gmail.com next Tuesday at 2pm for 1 hour
================================= Tool Message =================================
Name: schedule_event

Your meeting with **rrmalj@gmail.com** has been scheduled:

- **Date:** Tuesday, March 17, 2026  
- **Time:** 2:00 PM – 3:00 PM (1 hour)  
- **Title:** Meeting  

Let me know if you’d like to adjust anything or schedule anything else!
================================== Ai Message ==================================
Tool Calls:
  manage_email (call_da44d73908e04fb1a364a3b4)
 Call ID: call_da44d73908e04fb1a364a3b

- The supervisor recognizes this requires both calendar and email actions, calls `schedule_event` for the meeting, then calls `manage_email` for the reminder.
- Each sub-agent completes its task, and the supervisor synthesizes both results into a coherent response.

::: {.callout-tip}
Refer to the [LangSmith trace](https://smith.langchain.com/public/95cd00a3-d1f9-4dba-9731-7bf733fb6a3c/r) to see the detailed information flow for the above run, including individual chat model prompts and responses.
:::